# 様々な定点観測実装
初め、CrewAIで試したが、Agentは定型的な処理には向かないと判明したので、生のPythonで実装した（Chain系も不要）

In [ ]:
# .envというファイルから環境変数をロード
import os
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
# OPENAI_API_KEYのチェック
print(os.getenv("OPENAI_API_KEY").startswith("sk-proj-"))

In [ ]:
# Web記事の収集
import os
import sys
import requests
from pathlib import Path
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from openai import OpenAI

# 保存先ディレクトリをPathで定義
output_dir = Path("fpo_output")

# ディレクトリが存在しなければ作成
output_dir.mkdir(parents=True, exist_ok=True)

my_inputs = {
        'Topics': '政治、金融・経済、軍事',
        'URLs': [
            "https://www.47news.jp/",
            "https://www.unz.com/",
            "https://insiderpaper.com/",
            "https://www.zerohedge.com/",
            "https://consortiumnews.com/",
            "https://www.energyintel.com/",
            "https://tass.com/", 
            "https://www.rt.com/",
            "https://sputniknews.com/",
            "https://www.arabnews.com/",
            "https://www.timesofisrael.com/",
            "https://southfront.press/",
            "https://anna-news.info/",
            "https://www.stripes.com/",
            "https://www.military.com/",
            "https://www.militarytimes.com/",
        ]
}

htmlfilepaths=[]
txtfilepaths=[]
mdfilepaths=[]
md2filepaths=[]
results = []

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "ja,en-US;q=0.9,en;q=0.8",
    "Accept-Encoding": "gzip, deflate, br",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Connection": "keep-alive"
}

number = 0
urls = my_inputs['URLs']

for url in urls:
    number = number + 1
    
    # URLからHTMLを取得
    session = requests.Session()
    session.headers.update(headers)
    response = session.get(url)
    response.encoding = response.apparent_encoding  # 自動判定（高精度）
    text = response.text

    # HTMLファイル名生成
    parsed = urlparse(url)
    filename = str(number) + "_" + parsed.netloc.replace(".", "_") + ".html"
    filepath = output_dir / filename   
    
    # HTMLファイルに保存
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(text)
        htmlfilepaths.append(str(filepath))

for filepath in htmlfilepaths:
    path = Path(filepath)
    with path.open("r", encoding="utf-8") as f:
        # BeautifulSoupでHTMLを解析し、
        # 本文っぽい部分を抽出してテキストで返す
        soup = BeautifulSoup(f.read(), "html.parser")

        # script, style削除
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()
            
        # 本文っぽい部分だけ取得
        main = soup.body or soup
        temp = main.get_text(separator="\n", strip=True)

    # 中間ファイルに保存
    try:
        new_filepath = path.with_suffix(".txt")
        with new_filepath.open("w", encoding="utf-8") as f:
            f.write(temp)
            txtfilepaths.append(new_filepath)
    except Exception as e:
        print(f"Error writing file: {str(e)}", file=sys.stderr)

In [ ]:
# Web記事の翻訳と変換（MD化）
client = OpenAI()

for filepath in txtfilepaths:
    texts = []
    with open(filepath, "r", encoding="utf-8") as f:
         texts.append(f.read())

    # OpenAIに送信
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=f"""
        以下を日本語に翻訳しマークダウン形式に変換せよ：
        {texts}
        """
    )
    
    # 中間ファイルに保存
    try:
        new_filepath = Path(filepath).with_suffix(".md")
        with open(new_filepath, "w", encoding="utf-8") as f:
                f.write(response.output[0].content[0].text)
                mdfilepaths.append(new_filepath)
    except Exception as e:
        print(f"Error writing file: {str(e)}", file=sys.stderr)

In [ ]:
# Web記事のトリミング
client = OpenAI()

for filepath in mdfilepaths:
    texts = []
    with open(filepath, "r", encoding="utf-8") as f:
         texts.append(f.read())

    # OpenAIに送信
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=f"""
        以下はニュースサイトのトップページです。ヘッダ部やフッタ部などメインコンテンツ以外を削除して下さい。：
        {texts}
        """
    )
    
    # 中間ファイルに保存
    try:
        new_filepath = Path(filepath).with_suffix("")
        new_filepath = str(new_filepath) + "2.md"
        with open(new_filepath, "w", encoding="utf-8") as f:
                f.write(response.output[0].content[0].text)
                md2filepaths.append(new_filepath)
    except Exception as e:
        print(f"Error writing file: {str(e)}", file=sys.stderr)

In [ ]:
# Web記事をマージして要約
client = OpenAI()

texts = []

for filepath in md2filepaths:
    texts.append("\n\n" + str(filepath) + "\n\n")
    with open(filepath, "r", encoding="utf-8") as f:
         texts.append(f.read())

# ファイルに保存
try:
    filepath = Path(output_dir / "Temporary").with_suffix(".md")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write("".join(texts))
except Exception as e:
    print(f"Error writing file: {str(e)}", file=sys.stderr)
    
# OpenAIに送信
topics = my_inputs['Topics']
response = client.responses.create(
    model="gpt-4.1-mini",
    input=f"""
    「一極体制から多極化へのシフト」を意識しつつ、{topics} のニュースから情勢の分析レポートを作成せよ：
    {texts}
    """
)
    
# ファイルに保存
try:
    filepath = Path(output_dir / "Report").with_suffix(".md")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(response.output[0].content[0].text)
except Exception as e:
    print(f"Error writing file: {str(e)}", file=sys.stderr)